# NeuralAI v17 — Scale the base: 360M → SmolLM2-1.7B
Graduate the production demo model from SmolLM2-360M-Instruct to SmolLM2-1.7B-Instruct.
Backends are already pluggable (LM Studio / llmster), so this is a **config swap + finetune**, not a rewrite.

Run this notebook on a Colab **GPU** runtime (T4 is enough for QLoRA; A100/L4 for full fine-tune).

In [ ]:
import torch, os, json
from huggingface_hub import login
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| cuda:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a')
# login(os.getenv('HF_TOKEN'))  # uncomment if pushing adapter to HF

## 1. Install deps

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets sentencepiece lmstudio

## 2. Load base 1.7B (4-bit NF4 via bitsandbytes for QLoRA)
Swapping `360M` → `1.7B` is the only model-size change; everything downstream is identical.

In [ ]:
BASE = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')
print('loaded', BASE)

## 3. Attach LoRA (r=32, alpha=64) — mirrors v15 DPO recipe but larger base

In [ ]:
from peft import LoraConfig, get_peft_model
lora = LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 4. Load DPO dataset v15 (597 pairs) — reuse existing preference data
Upload `data/train_dpo_v15.jsonl` to the session or mount your HF repo.

In [ ]:
import json
# from google.colab import files
# uploaded = files.upload()  # select data/train_dpo_v15.jsonl
data = [json.loads(l) for l in open('train_dpo_v15.jsonl') if l.strip()]
print('pairs:', len(data))

## 5. Train (DPO) — 3 epochs, batch 4, grad-accum 8 on T4

In [ ]:
from trl import DPOTrainer, DPOConfig
cfg = DPOConfig(
    output_dir='./v17-1.7b-dpo',
    per_device_train_batch_size=4, gradient_accumulation_steps=8,
    num_train_epochs=3, learning_rate=5e-5, logging_steps=10,
    save_strategy='epoch', bf16=True,
)
trainer = DPOTrainer(model=model, ref_model=None, args=cfg, tokenizer=tok,
    train_dataset=data)
trainer.train()

## 6. Export + register as pluggable backend
Save adapter, then point LM Studio / llmster at the new GGUF or the HF adapter path. Update `services/webui_service.py` `BACKEND` config — no code rewrite.

In [ ]:
model.save_pretrained('./v17-1.7b-dpo')
tok.save_pretrained('./v17-1.7b-dpo')
print('saved adapter -> ./v17-1.7b-dpo')
# Next: convert to GGUF via llama.cpp, drop into LM Studio, set backend config.